# 10.3 Patch 怎麼變成向量？


上一節每個小塊有48個像素數字，文字模型卻希望每個位置用固定寬度的特徵表示。我們可以讓同一張「加權配方表」把每塊的48個值轉成8個值。這8個值組成一個向量，也就是一排數字。本節關注寬度如何改變，先不把隨機輸出叫作已學會的顏色或形狀。

前置是[10.2的patch形狀](https://birdhackor.github.io/tiny-perceptron-vlm/10.2.html)與[W.5的矩陣配方](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)。線性層Linear就是可訓練的加權變換，每個輸出由輸入乘相應權重再相加得到。把它應用在最後一軸，會對每個patch用同一套權重，而保持圖片數量與patch數量。這種將像素變為特徵的入口稱為patch embedding。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from tiny_perceptron.multimodal import scene, patchify

torch.manual_seed(0)
patches = patchify(scene()[None], 4)
embedding = nn.Linear(48, 8)
features = embedding(patches)
print("輸入", tuple(patches.shape))
print("輸出", tuple(features.shape))
print("矩陣", tuple(embedding.weight.shape))

輸入是1張圖的16個patch，每塊48數值。`torch.manual_seed(0)`固定產生隨機數的起點，稱為隨機種子，讓這段程式重新執行時使用相同的初始隨機權重；0是選定的種子編號，不是像素值或訓練次數。`nn.Linear(48,8)`定義48進8出的變換，內部矩陣形狀是`(8,48)`，每一行負責一個輸出。`features=embedding(patches)`將相同變換應用16次。輸出依序`(1,16,48)`、`(1,16,8)`、`(8,48)`；位置16沒有變，只有每個位置的描述寬度由48變8。

這個操作沒有把整張圖平均成一個值，也沒有把16個patch壓成1個patch。圖像中的不同區域仍有不同位置，後面可以分別被讀取。8個輸出數字的含義來自訓練目標，不一定有一項能直接叫「紅色程度」。隨機初始化時只知道數學可以執行，不能看見一個8維向量就斷言模型認得圓形。

與文字embedding相比，文字常從離散編號查一排數字，而這裡從連續像素經過線性變換產生一排數字；兩者都提供後面層可以處理的特徵寬度。像素如果一邊用0到1、一邊用0到255，輸出尺度也會不同，因此轉換範圍需要和訓練保持一致。

想像只有兩個輸入亮度[1,0]，一個輸出用權重[0.5,0.5]且偏置0，結果就是1×0.5+0×0.5=0.5。換輸入[0,1]仍得0.5，這個特定配方把兩種差異合併了。48轉8同樣是一種壓縮，可能保留任務需要的差異，也可能失去它；訓練要找適合的配方，不能把寬度變短本身當成理解。共享同一配方的好處是不同位置遇到相似局部像素時可用相同方法處理，後面再加入位置資訊，區分它們所在的地方。

練習只把輸出寬度8改16，先預測features變成`(1,16,16)`，矩陣變成`(16,48)`，patch數仍16，再執行核對。兩個16碰巧一樣，意義卻分別是位置數與每位置的特徵數；請在紙上標出每個軸，不要因為數值相同就把它們混為一個概念。
